In [1]:
!pip install transformers -q

In [2]:
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

import zipfile

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

For AST, we are using a smaller learning rate than our CNN because we are fine-tuning a pretrained transformer.

In [4]:
AST_CONFIG = {
    "model_name": "AST",
    "variant_name": "real_ast_finetune_fold1",
    "pretrained_model": "MIT/ast-finetuned-audioset-10-10-0.4593",

    "num_epochs": 10,
    "learning_rate": 1e-5,
    "batch_size": 8,
    "weight_decay": 1e-4,

    "freeze_backbone": False,

    "sample_rate": 16000,
    "num_classes": 50,

    "save_dir": "/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/AST"
}

In [5]:
class ESC50ASTDataset(Dataset):
    def __init__(self, dataframe, audio_dir, feature_extractor, sample_rate=16000):
        self.dataframe = dataframe.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.feature_extractor = feature_extractor
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        audio_path = os.path.join(self.audio_dir, row["filename"])
        label = int(row["target"])
        filename = row["filename"]

        waveform, sr = torchaudio.load(audio_path)

        # convert stereo to mono if needed
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # resample if needed
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)

        waveform = waveform.squeeze().numpy()

        inputs = self.feature_extractor(
            waveform,
            sampling_rate=self.sample_rate,
            return_tensors="pt"
        )

        input_values = inputs["input_values"].squeeze(0)

        return input_values, label, filename

In [6]:
def get_ast_train_test_loaders(
    dataframe,
    audio_dir,
    test_fold,
    config,
    num_workers=2
):
    feature_extractor = AutoFeatureExtractor.from_pretrained(
        config["pretrained_model"]
    )

    train_df = dataframe[dataframe["fold"] != test_fold].reset_index(drop=True)
    test_df = dataframe[dataframe["fold"] == test_fold].reset_index(drop=True)

    train_dataset = ESC50ASTDataset(
        train_df,
        audio_dir,
        feature_extractor,
        sample_rate=config["sample_rate"]
    )

    test_dataset = ESC50ASTDataset(
        test_df,
        audio_dir,
        feature_extractor,
        sample_rate=config["sample_rate"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, test_loader

In [7]:
def build_ast_from_config(config, device=DEVICE):
    id2label = {i: str(i) for i in range(config["num_classes"])}
    label2id = {str(i): i for i in range(config["num_classes"])}

    model = AutoModelForAudioClassification.from_pretrained(
        config["pretrained_model"],
        num_labels=config["num_classes"],
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )

    if config["freeze_backbone"]:
        for name, param in model.named_parameters():
            if "classifier" not in name:
                param.requires_grad = False

    model.to(device)
    return model

In [8]:
def run_model_ast(model, train_loader, test_loader, config, device=DEVICE):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    start_time = time.time()

    for epoch in range(config["num_epochs"]):
        model.train()

        train_loss_total = 0
        train_correct = 0
        train_total = 0

        for xb, yb, _ in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            outputs = model(input_values=xb, labels=yb)
            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * yb.size(0)

            preds = logits.argmax(dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        train_loss = train_loss_total / train_total
        train_acc = train_correct / train_total

        model.eval()

        test_loss_total = 0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for xb, yb, _ in test_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                outputs = model(input_values=xb, labels=yb)
                loss = outputs.loss
                logits = outputs.logits

                test_loss_total += loss.item() * yb.size(0)

                preds = logits.argmax(dim=1)
                test_correct += (preds == yb).sum().item()
                test_total += yb.size(0)

        test_loss = test_loss_total / test_total
        test_acc = test_correct / test_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"[AST] Epoch {epoch+1}/{config['num_epochs']} | "
            f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f} | "
            f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}"
        )

    results = {
        "model": config["model_name"],
        "variant": config["variant_name"],
        "pretrained_model": config["pretrained_model"],
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc": history["train_acc"][-1],
        "final_test_loss": history["test_loss"][-1],
        "final_test_acc": history["test_acc"][-1],
        "best_test_acc": max(history["test_acc"]),
        "best_epoch": int(np.argmax(history["test_acc"]) + 1),
        "training_time_sec": time.time() - start_time,
        "num_trainable_params": sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )
    }

    return history, results

In [9]:
def save_ast_results(history, results, config):
    save_path = (
        Path(config["save_dir"])
        / config["model_name"]
        / config["variant_name"]
    )
    save_path.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(history).to_csv(save_path / "history.csv", index=False)
    pd.DataFrame([results]).to_csv(save_path / "results.csv", index=False)

    with open(save_path / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    # Loss curve
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["test_loss"], label="Test Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{config['model_name']} - {config['variant_name']} Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path / "loss_curves.png", dpi=200, bbox_inches="tight")
    plt.close()

    # Accuracy curve
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_acc"], label="Train Acc")
    plt.plot(history["test_acc"], label="Test Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{config['model_name']} - {config['variant_name']} Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path / "accuracy_curves.png", dpi=200, bbox_inches="tight")
    plt.close()

    print("Saved AST results to:", save_path)

In [10]:
def run_ast_experiment(config, dataframe, audio_dir, test_fold, device=DEVICE):
    print(f"Running {config['model_name']} | {config['variant_name']} | fold {test_fold}")

    train_loader, test_loader = get_ast_train_test_loaders(
        dataframe=dataframe,
        audio_dir=audio_dir,
        test_fold=test_fold,
        config=config
    )

    model = build_ast_from_config(config, device=device)

    history, results = run_model_ast(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        config=config,
        device=device
    )

    save_ast_results(history, results, config)

    return history, results

In [11]:
ZIP_PATH = '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/piczak_dataset.zip'
EXTRACT_DIR = '/content/piczak_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/5888_Project/results/Phase1_Baseline'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
        zip_file.extractall(EXTRACT_DIR)

CSV_PATH = None
AUDIO_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'esc50.csv' in files:
        CSV_PATH = os.path.join(root, 'esc50.csv')

    wav_files = [file_name for file_name in files if file_name.endswith('.wav')]
    if len(wav_files) > 0 and AUDIO_DIR is None:
        AUDIO_DIR = root

print('Dataset CSV:  ', CSV_PATH)

df = pd.read_csv(CSV_PATH)

display(df.head())

Dataset CSV:   /content/piczak_dataset/esc50.csv


,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [12]:
history_ast, results_ast = run_ast_experiment(
    config=AST_CONFIG,
    dataframe=df,
    audio_dir=AUDIO_DIR,
    test_fold=1,
    device=DEVICE
)

results_ast

Running AST | real_ast_finetune_fold1 | fold 1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([50])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([50, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


[AST] Epoch 1/10 | Train Loss=1.8070, Train Acc=0.6975 | Test Loss=0.5834, Test Acc=0.9200
[AST] Epoch 2/10 | Train Loss=0.2040, Train Acc=0.9838 | Test Loss=0.3210, Test Acc=0.9325
[AST] Epoch 3/10 | Train Loss=0.0599, Train Acc=0.9969 | Test Loss=0.2616, Test Acc=0.9375
[AST] Epoch 4/10 | Train Loss=0.0277, Train Acc=0.9994 | Test Loss=0.2345, Test Acc=0.9375
[AST] Epoch 5/10 | Train Loss=0.0161, Train Acc=1.0000 | Test Loss=0.2355, Test Acc=0.9400
[AST] Epoch 6/10 | Train Loss=0.0111, Train Acc=1.0000 | Test Loss=0.2217, Test Acc=0.9425
[AST] Epoch 7/10 | Train Loss=0.0083, Train Acc=1.0000 | Test Loss=0.2160, Test Acc=0.9400
[AST] Epoch 8/10 | Train Loss=0.0065, Train Acc=1.0000 | Test Loss=0.2147, Test Acc=0.9400
[AST] Epoch 9/10 | Train Loss=0.0052, Train Acc=1.0000 | Test Loss=0.2159, Test Acc=0.9425
[AST] Epoch 10/10 | Train Loss=0.0043, Train Acc=1.0000 | Test Loss=0.2154, Test Acc=0.9400
Saved AST results to: /content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/AST/AST/r

{'model': 'AST',
 'variant': 'real_ast_finetune_fold1',
 'pretrained_model': 'MIT/ast-finetuned-audioset-10-10-0.4593',
 'final_train_loss': 0.004289460151921958,
 'final_train_acc': 1.0,
 'final_test_loss': 0.21537345730699597,
 'final_test_acc': 0.94,
 'best_test_acc': 0.9425,
 'best_epoch': 6,
 'training_time_sec': 337.7637367248535,
 'num_trainable_params': 86227250}

In [13]:
def run_ast_5_fold_evaluation(config, dataframe, audio_dir, device=DEVICE):
    all_results = []
    all_histories = {}

    for fold in sorted(dataframe["fold"].unique()):
        fold_config = config.copy()
        fold_config["variant_name"] = f"real_ast_finetune_fold{fold}"

        print("=" * 70)
        print(f"Running AST 5-fold evaluation: fold {fold}")
        print("=" * 70)

        history, results = run_ast_experiment(
            config=fold_config,
            dataframe=dataframe,
            audio_dir=audio_dir,
            test_fold=fold,
            device=device
        )

        results["fold"] = fold
        all_results.append(results)
        all_histories[f"fold_{fold}"] = history

    results_df = pd.DataFrame(all_results)

    summary = {
        "model": config["model_name"],
        "pretrained_model": config["pretrained_model"],
        "num_folds": len(results_df),
        "mean_test_acc": results_df["final_test_acc"].mean(),
        "std_test_acc": results_df["final_test_acc"].std(),
        "mean_best_test_acc": results_df["best_test_acc"].mean(),
        "std_best_test_acc": results_df["best_test_acc"].std(),
        "mean_test_loss": results_df["final_test_loss"].mean(),
        "std_test_loss": results_df["final_test_loss"].std(),
    }

    save_path = (
        Path(config["save_dir"])
        / config["model_name"]
        / "real_ast_5fold_summary"
    )
    save_path.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(save_path / "fold_results.csv", index=False)
    pd.DataFrame([summary]).to_csv(save_path / "summary.csv", index=False)

    with open(save_path / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\n5-Fold AST Summary")
    print("------------------")
    print(f"Mean Final Test Accuracy: {summary['mean_test_acc']:.4f}")
    print(f"Std Final Test Accuracy:  {summary['std_test_acc']:.4f}")
    print(f"Mean Best Test Accuracy:  {summary['mean_best_test_acc']:.4f}")
    print(f"Std Best Test Accuracy:   {summary['std_best_test_acc']:.4f}")
    print(f"Saved summary to: {save_path}")

    return results_df, summary, all_histories

In [14]:
ast_5fold_results, ast_5fold_summary, ast_5fold_histories = run_ast_5_fold_evaluation(
    config=AST_CONFIG,
    dataframe=df,
    audio_dir=AUDIO_DIR,
    device=DEVICE
)

Running AST 5-fold evaluation: fold 1
Running AST | real_ast_finetune_fold1 | fold 1


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([50])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([50, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


[AST] Epoch 1/10 | Train Loss=1.8155, Train Acc=0.6844 | Test Loss=0.5749, Test Acc=0.9175
[AST] Epoch 2/10 | Train Loss=0.2236, Train Acc=0.9838 | Test Loss=0.3174, Test Acc=0.9500
[AST] Epoch 3/10 | Train Loss=0.0634, Train Acc=0.9975 | Test Loss=0.2742, Test Acc=0.9425
[AST] Epoch 4/10 | Train Loss=0.0286, Train Acc=0.9994 | Test Loss=0.2198, Test Acc=0.9550
[AST] Epoch 5/10 | Train Loss=0.0168, Train Acc=1.0000 | Test Loss=0.2166, Test Acc=0.9575
[AST] Epoch 6/10 | Train Loss=0.0118, Train Acc=1.0000 | Test Loss=0.2108, Test Acc=0.9550
[AST] Epoch 7/10 | Train Loss=0.0088, Train Acc=1.0000 | Test Loss=0.2039, Test Acc=0.9525
[AST] Epoch 8/10 | Train Loss=0.0069, Train Acc=1.0000 | Test Loss=0.2021, Test Acc=0.9550
[AST] Epoch 9/10 | Train Loss=0.0055, Train Acc=1.0000 | Test Loss=0.2015, Test Acc=0.9550
[AST] Epoch 10/10 | Train Loss=0.0045, Train Acc=1.0000 | Test Loss=0.1987, Test Acc=0.9550
Saved AST results to: /content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/AST/AST/r

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

...

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([50])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([50, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


[AST] Epoch 1/10 | Train Loss=1.8171, Train Acc=0.7056 | Test Loss=0.4875, Test Acc=0.9450
[AST] Epoch 2/10 | Train Loss=0.2128, Train Acc=0.9825 | Test Loss=0.2667, Test Acc=0.9525
[AST] Epoch 3/10 | Train Loss=0.0635, Train Acc=0.9975 | Test Loss=0.1841, Test Acc=0.9525
[AST] Epoch 4/10 | Train Loss=0.0305, Train Acc=0.9988 | Test Loss=0.1614, Test Acc=0.9600
[AST] Epoch 5/10 | Train Loss=0.0165, Train Acc=1.0000 | Test Loss=0.1452, Test Acc=0.9575
[AST] Epoch 6/10 | Train Loss=0.0114, Train Acc=1.0000 | Test Loss=0.1397, Test Acc=0.9550
[AST] Epoch 7/10 | Train Loss=0.0085, Train Acc=1.0000 | Test Loss=0.1297, Test Acc=0.9550
[AST] Epoch 8/10 | Train Loss=0.0066, Train Acc=1.0000 | Test Loss=0.1244, Test Acc=0.9550
[AST] Epoch 9/10 | Train Loss=0.0053, Train Acc=1.0000 | Test Loss=0.1224, Test Acc=0.9550
[AST] Epoch 10/10 | Train Loss=0.0043, Train Acc=1.0000 | Test Loss=0.1214, Test Acc=0.9550
Saved AST results to: /content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/AST/AST/r

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([50])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([50, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


[AST] Epoch 1/10 | Train Loss=1.7430, Train Acc=0.7044 | Test Loss=0.6410, Test Acc=0.8925
[AST] Epoch 2/10 | Train Loss=0.2038, Train Acc=0.9825 | Test Loss=0.3822, Test Acc=0.9075
[AST] Epoch 3/10 | Train Loss=0.0686, Train Acc=0.9956 | Test Loss=0.3074, Test Acc=0.9175
[AST] Epoch 4/10 | Train Loss=0.0283, Train Acc=0.9988 | Test Loss=0.2775, Test Acc=0.9250
[AST] Epoch 5/10 | Train Loss=0.0165, Train Acc=1.0000 | Test Loss=0.2717, Test Acc=0.9200
[AST] Epoch 6/10 | Train Loss=0.0111, Train Acc=1.0000 | Test Loss=0.2446, Test Acc=0.9175
[AST] Epoch 7/10 | Train Loss=0.0083, Train Acc=1.0000 | Test Loss=0.2510, Test Acc=0.9250
[AST] Epoch 8/10 | Train Loss=0.0065, Train Acc=1.0000 | Test Loss=0.2457, Test Acc=0.9250
[AST] Epoch 9/10 | Train Loss=0.0052, Train Acc=1.0000 | Test Loss=0.2398, Test Acc=0.9250
[AST] Epoch 10/10 | Train Loss=0.0042, Train Acc=1.0000 | Test Loss=0.2366, Test Acc=0.9225
Saved AST results to: /content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/AST/AST/r